# Lecture 10: Continuation: from easy problems to hard ones

**The challenge.** Lecture 9's random search worked on the cart-pole
because about one guess in eight fell into Newton's basin. Now take a
spacecraft with a weak engine raising its orbit over two and a half
revolutions, spending as little propellant as possible. The throttle is
bang-bang (Lecture 4): coast, burn, coast, burn. Change the costate guess a
little and a switch appears or disappears, and the residual jumps. Newton's
basin is tiny and ragged. Below, one random guess in sixty converges. For a
real mission, with more revolutions, it is none.

The way out is not to solve the hard problem directly. Solve an *easy*
problem that is close to it, then deform the easy problem into the hard one
in small steps, re-solving at each step from the previous solution. This is
**continuation**, also called homotopy. It is the single most useful idea in
the practice of the indirect method.

**After this lecture you can**

- set up a homotopy between an easy problem and a hard one, and follow it
  with natural-parameter continuation;
- smooth a bang-bang problem with `regularize` and continue the smoothing
  to zero;
- continue in a physical parameter with `parameter_continuation`, and
  recognize when a path breaks.

## 1. Deforming one problem into another

The idea is old. Henri Poincaré, studying periodic orbits of the three-body
problem in the 1890s, followed them as a parameter (a mass ratio) changed
from a value where they were known, the *method of continuity*. As a
numerical method for systems of equations it appears in Edmond Lahaye's work
in 1934. Dmitrii Davidenko observed in 1953 that the solution of
$R(v, s) = 0$ along the parameter $s$ obeys an ordinary differential
equation, $\mathrm{d}v/\mathrm{d}s = -R_v^{-1} R_s$, so following a path is
just another integration. Herbert Keller's *pseudo-arclength* method of 1977
followed paths around the turning points where natural-parameter
continuation fails.

In optimal control, continuation became standard for orbit transfers in the
early 2000s, much of it in Toulouse, around the French space agency CNES and
the ENSEEIHT engineering school. Bertrand and Epenoy (2002), at CNES, smoothed the bang-bang throttle with a
penalty that vanishes in the limit, and Caillau, Gergaud and Noailles (2003)
computed transfers to geostationary orbit with thrust as low as 0.1 N by
continuing in the thrust level itself. Both ideas are in this lecture.

In [ ]:
# Continuation on a toy path, R(v, s) = v^3 - 3 v - s = 0, from s = -3. Stepping s and
# re-solving (natural-parameter continuation) works until the turning point at s = 2, where
# the branch it follows ends. Following arclength goes around the turn.
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import fsolve

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
v_curve = np.linspace(-2.3, 2.3, 400)
s_curve = v_curve**3 - 3 * v_curve
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(s_curve, v_curve, color=GRAY, lw=1.5, label="the solution path R(v, s) = 0")
v, natural = -2.1038, []
for s in np.arange(-3.0, 1.95, 0.35):
    v = fsolve(lambda x: x**3 - 3 * x - s, v)[0]
    natural.append((s, v))
natural = np.array(natural)
ax.plot(*natural.T, "o", color=ORANGE, ms=7, label="natural steps in s")
ax.plot(2.25, -1.0, "x", color=ORANGE, ms=12, mew=3)
ax.annotate("s = 2.25: no solution nearby,\nthe step fails", (2.25, -1.0), xytext=(2.4, -2.0),
            fontsize=9, arrowprops={"arrowstyle": "-|>", "lw": 0.8})
arc = np.linspace(-1.0, 1.0, 9)
ax.plot(arc**3 - 3 * arc, arc, "o", color=BLUE, ms=5, label="arclength steps, around the turn")
ax.plot([2], [-1], "o", ms=14, mfc="none", mec="k")
ax.text(2.1, -0.75, "turning point", fontsize=9)
ax.set(xlabel="continuation parameter s", ylabel="solution v")
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.grid(alpha=0.25)
fig.tight_layout()

*Continuation on a toy equation $v^3 - 3v - s = 0$. Stepping $s$ and solving from the previous solution follows the lower branch until it ends at a turning point, where the next step has no solution nearby. Keller's arclength continuation steps along the curve instead, and goes around the turn.*

## 2. The hard problem

`planar_low_thrust_fuel` transfers a spacecraft from a circular orbit of
radius $r_0 = 1$ to one of radius $r_f = 1.5$ (normalized units,
$\mu = 1$) in time $t_f = 15$, with thrust $T_{\max} = 0.02$ and exhaust
speed $c = 1$, minimizing the propellant $\int T_{\max}\delta/c$. The
throttle $\delta \in [0, 1]$ is bang-bang and the thrust direction is the
primer vector (Lecture 4). The terminal conditions are three: on the target
circle, with zero radial and circular tangential velocity, at any phase.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from auto_optimal_control import examples
from auto_optimal_control.build import build_kernels
from auto_optimal_control.homotopy import regularize
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.shooting import SingleShooting, continuation, parameter_continuation

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

fuel = examples.planar_low_thrust_fuel()
k_fuel = build_kernels(derive(fuel))
p = [1.0, 0.02, 1.0, 1.0, 1.0, 1.5, 15.0]            # mu, T_max, c, m0, r0, rf, tf
box = ([-2, -2, -2, -2, 0], [2, 2, 2, 2, 2])         # where to draw lam0 from


def success_rate(kernels, params, n=60, seed=0):
    shoot = SingleShooting(kernels, params)
    rng = np.random.default_rng(seed)
    ok = 0
    for _ in range(n):
        try:
            ok += shoot.solve(shoot.guess_multipliers(rng.uniform(*box))).success
        except Exception:
            pass
    return ok, n


start = time.perf_counter()
print("fuel-optimal, random guesses: %d of %d converge" % success_rate(k_fuel, p),
      f"({time.perf_counter() - start:.1f} s)")

## 3. The easy problem: smooth the throttle

Lecture 4's exercise added $\varepsilon u^2/2$ to a linear cost and turned the
bang-bang law into a clipped one. `homotopy.regularize` does that for every
bang-bang control of a problem, scaled to the control's range:

$$
L_\varepsilon = L + \varepsilon\,\Big(\frac{u - m}{h}\Big)^2\Big/2,
$$

with $m$ and $h$ the middle and half-width of $[u_{\text{lo}}, u_{\text{hi}}]$.
The law becomes $u = \operatorname{clip}(m - h^2 S/\varepsilon)$: at
$\varepsilon = 1$ a gentle ramp, as $\varepsilon \to 0$ the step of the
bang-bang law. $\varepsilon$ is a new runtime parameter, `eps_reg`, at the end
of the parameter vector. (It also adds a penalty weight for each path
constraint, Lecture 13; this problem has none.)

In [ ]:
# The smoothed throttle law, delta = clip(m - h^2 S / eps, 0, 1) with m = h = 1/2, against
# the switching function: a ramp that steepens into the bang-bang step as eps falls.
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
S = np.linspace(-1.5, 1.5, 1001)
fig, ax = plt.subplots(figsize=(7, 3.4))
cmap = plt.get_cmap("Blues")
for i, e in enumerate([3.0, 1.0, 0.3, 0.1, 0.03]):
    ax.plot(S, np.clip(0.5 - 0.25 * S / e, 0, 1), color=cmap(0.35 + 0.15 * i), lw=2,
            label=rf"$\varepsilon$ = {e:g}")
ax.plot(S, np.where(S < 0, 1.0, 0.0), color=ORANGE, lw=2, ls="--", label=r"bang-bang ($\varepsilon \to 0$)")
ax.set(xlabel="switching function S", ylabel=r"throttle $\delta$")
ax.legend(frameon=False, fontsize=8, loc="upper right")
ax.set_title("the smoothed throttle law", fontsize=10, loc="left")
ax.grid(alpha=0.25)
fig.tight_layout()

*The smoothed law as a function of the switching function. At large $\varepsilon$ it is a gentle ramp, and the residual is smooth. As $\varepsilon$ falls, the ramp steepens into the step of the bang-bang law.*

In [ ]:
smooth, eps, rhos = regularize(fuel)
print("added parameters:", [str(q) for q in smooth.params[len(fuel.params):]])
k_smooth = build_kernels(derive(smooth))
print("throttle law of the smoothed problem:", derive(smooth).control_law[fuel.controls[2]])
print("smoothed (eps = 1), random guesses: %d of %d converge" % success_rate(k_smooth, p + [1.0]))

Half the guesses now converge. A continuous throttle means a residual without
jumps, and a much larger basin for Newton.

## 4. Following the path

Natural-parameter continuation is the simplest scheme. Step the parameter
along a list of values, solve each problem from the previous solution, and
when a solve fails, try a value halfway back. `continuation(solve_at, start,
values, v0)` does exactly that for any `solve_at(value, guess)`;
`parameter_continuation` specializes it to one runtime parameter of a
`SingleShooting`.

In [ ]:
shoot = SingleShooting(k_smooth, p + [1.0])
first, tries = shoot.multistart(lambda rng: shoot.guess_multipliers(rng.uniform(*box)),
                                n_tries=50, seed=0)
start = time.perf_counter()
path = parameter_continuation(k_smooth, p + [1.0], "eps_reg", np.logspace(0, -4, 17), first.v)
print(f"{len(path)} solves from eps = 1 to {path[-1][0]:.0e} in {time.perf_counter() - start:.1f} s")

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.4))
for (e, sol), color in zip([path[0], path[4], path[8], path[-1]],
                           [GRAY, ORANGE, BLUE, "k"]):
    tr = SingleShooting(k_smooth, p + [e]).trajectory(sol.v, n_points=1500)
    left.plot(tr["t"], tr["u"][:, 2], color=color, lw=1.5, label=f"eps = {e:.0e}")
left.set(xlabel="t", ylabel=r"throttle $\delta$")
left.legend(frameon=False, fontsize=8)
costs = [k_smooth.propagate(0.0, s.z0, 15.0, p + [e], cost=True)["cost"] for e, s in path]
right.semilogx([e for e, _ in path], costs, "o-", color=BLUE, ms=4)
right.set(xlabel="eps", ylabel=r"cost $\int L_\varepsilon$")
for ax in (left, right):
    ax.grid(alpha=0.3)
fig.tight_layout()

As $\varepsilon$ falls, the throttle swings further and reaches its bounds,
and the cost converges. But even at $\varepsilon = 10^{-4}$ it is not yet
bang-bang: it still has broad, smooth humps. There the switching function
stays close to zero for a long time, so even a tiny $\varepsilon$ leaves the
throttle in between, a *nearly singular* structure (Lecture 12 is about the
exactly singular case). The smoothed problem and the fuel problem have the same
unknowns, so any smoothed solution is a starting point for the exact one.
Which one to jump from is less obvious than it looks. Try the exact solve
from every point of the path:

In [ ]:
k_exact = SingleShooting(k_fuel, p)
jumps = []
for e, sol in path:
    ex = k_exact.solve(sol.v)
    J = k_fuel.propagate(0.0, ex.z0, 15.0, p, cost=True)["cost"] if ex.success else np.nan
    jumps.append((e, ex, J))
    print(f"eps = {e:8.1e}: exact solve {'converges' if ex.success else 'fails    '}"
          + (f"  propellant {J:.8f}" if ex.success else ""))

Two things stand out. First, the jump does not get steadily easier as
$\varepsilon$ falls. The smoothed humps are a poor picture of the exact
problem's short, sharp burns, and the solve from $\varepsilon = 10^{-4}$
fails, while some jumps from much larger $\varepsilon$ succeed. Second,
different jumps land on **different extremals**: two propellant values
appear, both satisfying every necessary condition. As Lecture 5 warned, the
only way to rank them is by cost. The sensible policy is to attempt the exact
solve at every step and keep the cheapest success, which is what the
automatic homotopy of Lecture 15 does.

In [ ]:
e_best, exact, J_best = min((j for j in jumps if j[1].success), key=lambda j: j[2])
r = k_fuel.propagate(0.0, exact.z0, 15.0, p, cost=True)
print(f"cheapest: from eps = {e_best:.1e}, propellant = {J_best:.10f}, "
      f"|R| = {exact.residual_norm:.1e}")
print("switch times:", np.round(r["switch_times"], 5))

tr = SingleShooting(k_fuel, p).trajectory(exact.v, n_points=3000)
burn = tr["u"][:, 2] > 0.5
fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.plot(tr["z"][~burn, 0], tr["z"][~burn, 1], ".", color=GRAY, ms=1)
ax.plot(tr["z"][burn, 0], tr["z"][burn, 1], ".", color=ORANGE, ms=1.5)
th = np.linspace(0, 2 * np.pi, 300)
for rad in (1.0, 1.5):
    ax.plot(rad * np.cos(th), rad * np.sin(th), "k--", lw=0.5)
ax.set_aspect("equal")
ax.set_title("burn (orange) and coast (gray)", fontsize=10, loc="left")
ax.grid(alpha=0.3)
fig.tight_layout()

There is no burn schedule anywhere in the problem statement. The solver
placed the burns itself, from the costates alone, which is what the indirect
method buys: the structure of the answer comes out of the necessary
conditions.

## 5. Continuation in a physical parameter

The same machinery follows a family of problems. Raise the target radius
from 1.5 towards 1.8 on the exact fuel problem, solving each from the last:

In [ ]:
targets = np.linspace(1.5, 1.8, 13)
fam = parameter_continuation(k_fuel, p, "r_f", targets, exact.v, min_step=1e-3)
print(f"followed to r_f = {fam[-1][0]:.3f} in {len(fam)} solves")
fuel_used = [k_fuel.propagate(0.0, s.z0, 15.0, [*p[:5], rf, p[6]], cost=True)["cost"]
             for rf, s in fam]
n_sw = [len(k_fuel.propagate(0.0, s.z0, 15.0, [*p[:5], rf, p[6]])["switch_times"])
        for rf, s in fam]
for rf, J, n in list(zip([f[0] for f in fam], fuel_used, n_sw))[::3]:
    print(f"r_f = {rf:.3f}: propellant {J:.6f}, {n} switches")

The propellant rises smoothly with the target radius. The switch structure
did *not* stay fixed: between $r_f = 1.5$ and $1.575$ one switch pair merged
and a short arc vanished, and the continuation followed anyway, because the
shooting unknowns (the costates) do not care how many switches there are;
the integration finds them. That is a real advantage of shooting on the
costates over methods that fix the structure in advance. Two things can still
break such a path, and it is worth knowing what they look like:

- **The structure changes badly.** When an arc shrinks to nothing, the exact
  bang-bang residual has a kink at that parameter value, and Newton may fail
  right there. The remedy is to continue the *smoothed* problem through the
  change instead, then jump again.
- **A turning point.** The path folds back: past some value there is no
  nearby solution. Natural-parameter continuation then shrinks its step until
  `continuation` gives up with `RuntimeError`; pseudo-arclength methods,
  which parametrize the path by its length, go around.

A good predictor helps any continuation. `continuation` starts each solve
from the previous solution; the automatic homotopy of Lecture 15 uses a
secant predictor and adapts its step, and with the parameter sensitivity of
Lecture 8 one can predict along the tangent
$\mathrm{d}v/\mathrm{d}s = -R_v^{-1}R_s$, as Davidenko proposed.

## Summary

- Continuation solves a hard problem by deforming an easy one into it,
  re-solving at each step from the last solution.
- For bang-bang controls, `regularize` adds a vanishing quadratic smoothing
  (parameter `eps_reg`); the smoothed problem converges from far more
  guesses, and its solution at small $\varepsilon$ seeds the exact problem,
  which has the same unknowns.
- `continuation` and `parameter_continuation` follow a path with step
  halving; they fail at turning points and can stumble where the switch
  structure changes.

## Exercises

1. Start the $\varepsilon$ path at 10 instead of 1. Does the random search
   get easier still? Does the path get longer?
2. Continue the fuel problem in the thrust level, $T_{\max}$ from 0.02 down
   to 0.01, and plot the number of switches against $T_{\max}$. Where does
   the path have trouble, and why?
3. Replace the quadratic smoothing by Bertrand and Epenoy's logarithmic
   barrier, $L - \varepsilon\,[\ln\delta + \ln(1 - \delta)]$, as your own
   `OptimalControlProblem`. Compare the continuation paths.

## Further reading

- E. L. Allgower and K. Georg, *Introduction to Numerical Continuation
  Methods* (SIAM, 2003), chapters 1 and 2: natural parameter and
  pseudo-arclength continuation.
- R. Bertrand and R. Epenoy, "New smoothing techniques for solving bang-bang
  optimal control problems", *Optimal Control Applications and Methods* 23,
  2002.
- J.-B. Caillau, J. Gergaud and J. Noailles, "3D geosynchronous transfer of
  a satellite: continuation on the thrust", *Journal of Optimization Theory
  and Applications* 118, 2003.